# Massachusetts Town Geometries

Pulls Massachusetts town (county subdivision) boundary geometries from the
Census TIGER/Line dataset and plots Brookline on a contextily basemap. The
`towns` GeoDataFrame built below covers every MA town, so it's reusable for
other municipal-boundary views beyond this one example.

In [ ]:
from pathlib import Path
from zipfile import ZipFile

import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
import requests
import xyzservices
import yaml

from shared.plots.style import apply_theme, add_source_footnote

apply_theme()

## Load or fetch raw geometries

`data/raw/tl_2022_25_cousub/tl_2022_25_cousub.zip` is the TIGER/Line 2022
county-subdivision shapefile for Massachusetts (state FIPS 25) — county
subdivisions are Massachusetts's towns/cities. A re-run loads the cached
zip straight from disk; only a missing cache triggers a download.

In [ ]:
TIGER_URL = "https://www2.census.gov/geo/tiger/TIGER2022/COUSUB/tl_2022_25_cousub.zip"

raw_dir = Path("data/raw/tl_2022_25_cousub")
zip_path = raw_dir / "tl_2022_25_cousub.zip"

if zip_path.exists():
    print(f"Loaded cached pull: {zip_path}")
else:
    raw_dir.mkdir(parents=True, exist_ok=True)
    response = requests.get(TIGER_URL, timeout=60)
    response.raise_for_status()
    zip_path.write_bytes(response.content)
    print(f"Fetched and cached: {zip_path}")

shp_name = next(
    name for name in ZipFile(zip_path).namelist() if name.endswith(".shp")
)
towns = gpd.read_file(f"zip://{zip_path}!{shp_name}")
towns.head()

## Plot: Brookline on a contextily basemap

Uses the CARTO Positron basemap per the `map-artifacts` skill, with the
CARTO API key from `config_secrets.yml` (repo root) appended to the tile
URL to avoid the anonymous-tile watermark.

In [ ]:
def load_carto_api_key():
    path = Path("../../../config_secrets.yml")
    if not path.exists():
        print("Warning: config_secrets.yml not found; CARTO map tiles may be rate-limited.")
        return None
    key = (yaml.safe_load(path.read_text()) or {}).get("api_keys", {}).get("carto")
    if not key:
        print("Warning: no CARTO api key set in config_secrets.yml; map tiles may be rate-limited.")
    return key


carto_api_key = load_carto_api_key()

basemap_source = dict(ctx.providers.CartoDB.Positron)
if carto_api_key:
    basemap_source["url"] += "?key={api_key}"
    basemap_source["api_key"] = carto_api_key
basemap_source = xyzservices.TileProvider(basemap_source)

brookline = towns[towns["NAME"] == "Brookline"].to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(8, 8))
brookline.plot(ax=ax, facecolor="#2A6F97", edgecolor="#1B4965", alpha=0.5, linewidth=2)

ctx.add_basemap(ax, source=basemap_source)
ax.set_axis_off()
ax.set_title("Brookline, Massachusetts")

add_source_footnote(fig, "US Census TIGER/Line 2022 (COUSUB, Massachusetts)")

output_dir = Path("outputs/charts")
output_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(output_dir / "brookline_map.png", bbox_inches="tight")
plt.show()